In [13]:
import polars as pl
import json

lazy_df = pl.scan_csv("../Data/ha_processed_data.csv")

def is_valid(col_name):
    return (
        pl.col(col_name).is_not_null() & 
        (pl.col(col_name).str.to_lowercase() != "none") & 
        (pl.col(col_name).str.strip_chars() != "")
    )

# 3. Transform and Aggregate
test_df = (
    lazy_df
    .with_columns([
        pl.struct(
            degree_user_input = pl.when(pl.col("degree_raw").is_null()).then(None).otherwise(pl.col("degree")),
            degree_standardized = pl.col("degree_raw"),
            university = pl.col("university"),
            major_user_input = pl.when(pl.col("major").is_null()).then(None).otherwise(pl.col("field")),
            major_standardized = pl.col("major"),
            description = pl.col("description_edu"),
            start_date = pl.col("start_year"),
            end_date = pl.col("end_year")
        ).alias("edu_struct"),
        
        pl.struct(
            title = pl.col("title"),
            company = pl.col("company"),
            industry_user_input = pl.col("industry"),
            industry_standardized = pl.col("industry standard"),
            description = pl.col("description_exp"),
            start_date = pl.col("start_date"),
            end_date = pl.col("end_date"),
            skillset = pl.col("skillset")
        ).alias("job_struct")
    ])
    .group_by("id")
    .agg([
        pl.col("first_industry").filter(is_valid("first_industry")).first().alias("first_industry"),
        pl.col("current_industry").filter(is_valid("current_industry")).first().alias("current_industry"),
        
        # --- Education: Filter out all-null structs before unique ---
        pl.col("edu_struct")
            .filter(
                pl.col("edu_struct").struct.field("university").is_not_null() | 
                pl.col("edu_struct").struct.field("degree_standardized").is_not_null()
            )
            .unique()
            .alias("educations"),
            
        # --- Jobs: Filter out all-null structs before unique ---
        pl.col("job_struct")
            .filter(
                pl.col("job_struct").struct.field("title").is_not_null() & 
                pl.col("job_struct").struct.field("company").is_not_null()
            )
            .unique()
            .alias("jobs")
    ])
    .rename({"id": "id"})
    .collect(engine="streaming")
)

# ... rest of the serialization logic remains the same
final_test_df = test_df.with_columns([
    pl.col("educations").map_elements(lambda x: json.dumps(x.to_list()) if x is not None else "[]", return_dtype=pl.String),
    pl.col("jobs").map_elements(lambda x: json.dumps(x.to_list()) if x is not None else "[]", return_dtype=pl.String)
])

# Final Write
final_test_df.write_csv("../Data/merged_data.csv")

In [15]:
import polars as pl
import json
import ast

# --- Configuration ---
NAM_CSV = "../Data/nam_processed_data.csv"
AGGREGATED_CSV = "../Data/merged_data.csv"
OUTPUT_CSV = "../Data/final.csv"

NAM_COLS = [
    "id", "name", "city", "about", "recommendations_count",
    "courses", "languages", "certifications", "recommendations",
    "volunteer_experience", "followers", "connections", 
    "publications", "patents", "projects", "organizations",
    "education" 
]

AGG_COLS = [
    "id", "first_industry", "current_industry", "educations", "jobs"
]

# --- Logic Functions ---

def _transform_nam_edu_logic(json_str: str) -> str:
    """Converts NAM schema to Aggregated schema."""
    if not json_str or json_str in ("None", "", "null"):
        return "[]"
    try:
        data = json.loads(json_str)
        if not isinstance(data, list):
            return "[]"
        
        transformed = [
            {
                "degree_user_input": None,
                "degree_standardized": None,
                "university": entry.get("title"),
                "major_user_input": None,
                "major_standardized": None,
                "description": entry.get("description"),
                "start_date": entry.get("start_year"),
                "end_date": entry.get("end_year")
            }
            for entry in data
        ]
        return json.dumps(transformed)
    except Exception:
        return "[]"

def _scrub_education_entries(json_str: str) -> str:
    """Parses education and filters out only strictly empty entries."""
    if not json_str or json_str in ("None", "", "null") or json_str == "[]":
        return "EMPTY_EDU"
    try:
        data = json.loads(json_str)
        if not isinstance(data, list) or len(data) == 0:
            return "EMPTY_EDU"
        
        valid_entries = []
        
        for entry in data:
            univ = str(entry.get("university") or "").strip()
            if univ:
                valid_entries.append(entry)
        
        if not valid_entries:
            return "EMPTY_EDU"
            
        return json.dumps(valid_entries)
    except Exception:
        return "EMPTY_EDU"

def _scrub_jobs_entries(json_str: str) -> str:
    """Standardizes jobs and purges nested string-literal 'unknown' skillsets."""
    if not json_str or json_str in ("None", "", "null"):
        return "[]"
    try:
        data = json.loads(json_str)
        if not isinstance(data, list):
            return "[]"
        
        valid_entries = []
        for entry in data:
            title = str(entry.get("title") or "").strip()
            company = str(entry.get("company") or "").strip()
            
            if title and company:
                skills = entry.get("skillset", [])
                
                # Handle nested string literal lists: ["['item1', 'item2']"]
                if isinstance(skills, list) and len(skills) == 1:
                    first_val = str(skills[0]).strip()
                    if first_val.startswith("[") and first_val.endswith("]"):
                        try:
                            skills = ast.literal_eval(first_val)
                        except:
                            pass 

                # Standard Cleanup & Filtering
                if isinstance(skills, (list, str)):
                    skill_list = skills if isinstance(skills, list) else [skills]
                    clean_skills = []
                    for s in skill_list:
                        s_clean = str(s).strip().lower()
                        # Remove bracket/quote artifacts
                        s_clean = s_clean.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
                        
                        if s_clean and s_clean not in ("unknown", "none", "null"):
                            clean_skills.append(s_clean)
                    
                    entry["skillset"] = clean_skills
                else:
                    entry["skillset"] = []

                valid_entries.append(entry)
        
        return json.dumps(valid_entries)
    except Exception:
        return "[]"

def _clean_projects_logic(json_str: str) -> str:
    if not json_str or json_str in ("None", "", "null"):
        return "[]"
    try:
        data = json.loads(json_str)
        if not isinstance(data, list):
            return "[]"
        valid = [p for p in data if p.get("title") or p.get("description")]
        return json.dumps(valid)
    except Exception:
        return "[]"

# --- Pipeline ---

def load_and_join(nam_path: str, agg_path: str) -> pl.LazyFrame:
    nam_lf = pl.scan_csv(nam_path).select(NAM_COLS)
    agg_lf = pl.scan_csv(agg_path).select(AGG_COLS)
    return agg_lf.join(nam_lf, on="id", how="left")

def process_transformations(lf: pl.LazyFrame) -> pl.LazyFrame:
    is_agg_empty = (pl.col("educations").is_null()) | (pl.col("educations").str.len_chars() < 5)

    return lf.with_columns(
        pl.when(is_agg_empty)
        .then(pl.col("education").map_elements(_transform_nam_edu_logic, return_dtype=pl.String))
        .otherwise(pl.col("educations"))
        .alias("educations")
    ).with_columns([
        pl.col("educations").map_elements(_scrub_education_entries, return_dtype=pl.String),
        pl.col("projects").map_elements(_clean_projects_logic, return_dtype=pl.String),
        pl.col("jobs").map_elements(_scrub_jobs_entries, return_dtype=pl.String)
    ]).drop("education")

def main():
    print("Starting pipeline...")
    
    # 1. Load and Join
    lf = load_and_join(NAM_CSV, AGGREGATED_CSV)
    df = lf.collect()
    initial_count = df.height
    print(f"Initial records: {initial_count}")

    # 2. Transformations
    df = process_transformations(df.lazy()).collect()
    
    # 3. Filtering and Logging
    edu_empty_df = df.filter(pl.col("educations") == "EMPTY_EDU")
    
    # Log specific IDs for debugging
    if edu_empty_df.height > 0:
        print(f"Sample IDs dropped (Empty Edu): {edu_empty_df.select('id').head(5).to_series().to_list()}")

    print(f"Filtered out {edu_empty_df.height} records due to empty education.")

    # Apply filter: only remove records that are strictly empty
    df = df.filter(pl.col("educations") != "EMPTY_EDU")

    # Jobs Filtering
    before_jobs = df.height
    df = df.filter(pl.col("jobs") != "[]")
    
    print(f"Filtered out {before_jobs - df.height} records due to empty jobs.")
    
    # Final Stats
    print(f"Final dataset size: {df.height} (Total dropped: {initial_count - df.height})")

    # 4. Writing Output
    df.write_csv(OUTPUT_CSV)
    print(f"Success. Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Starting pipeline...
Initial records: 568061
Sample IDs dropped (Empty Edu): ['hong-le-thi-my-89307511', 'nguyen-ngochien-46b29111a', 'vo-thengan-668192222', 'phạm-vũ-6a493b195', 'ma-theresa-ibo-49968b17']
Filtered out 1317 records due to empty education.
Filtered out 336 records due to empty jobs.
Final dataset size: 566402 (Total dropped: 1659)
Success. Data saved to ../Data/final.csv


In [20]:
import polars as pl
import json
import sys

def get_person_by_index(file_path, target_index):
    try:
        # scan_csv creates a lazy pointer to the file
        lf = pl.scan_csv(file_path)
        
        # slice(offset, length) fetches exactly one row at the target index
        df = lf.slice(target_index, 1).collect()
        
        if df.is_empty():
            print(f"Error: Index {target_index} is out of range.")
            return

        # Convert the single row to a dictionary
        record = df.to_dicts()[0]

        print(f"--- Record at Index: {target_index} ---")
        
        for key, value in record.items():
            print(f"{key}:")
            
            # Check if the value is a string that looks like JSON
            if isinstance(value, str) and (value.startswith('{') or value.startswith('[')):
                try:
                    # Parse and beautify JSON
                    parsed_json = json.loads(value)
                    beautified_json = json.dumps(parsed_json, indent=4)
                    print(beautified_json)
                except json.JSONDecodeError:
                    # If parsing fails, print as raw text
                    print(value)
            else:
                # Print scalar text or other types as is
                print(value)
            
            print("-" * 13)
            print()

    except Exception as e:
        print(f"An error occurred: {e}")

CSV_PATH = "../Data/final.csv"

try:
    idx = 20
    get_person_by_index(CSV_PATH, idx)
except ValueError:
    print("Please provide a valid integer for the index.")

--- Record at Index: 20 ---
id:
huyen-ngg-b3499a227
-------------

first_industry:
education
-------------

current_industry:
education
-------------

educations:
[
    {
        "degree_user_input": null,
        "degree_standardized": null,
        "university": "Banking University of Ho Chi Minh City",
        "major_user_input": null,
        "major_standardized": null,
        "description": null,
        "start_date": null,
        "end_date": null
    }
]
-------------

jobs:
[
    {
        "title": "banker",
        "company": "bidv - ng\u00e2n h\u00e0ng tmcp \u0111\u1ea7u t\u01b0 & ph\u00e1t tri\u1ec3n vi\u1ec7t nam",
        "industry_user_input": "financial services",
        "industry_standardized": "activities auxiliary to financial service and insurance activities",
        "description": null,
        "start_date": null,
        "end_date": null,
        "skillset": [
            "banking"
        ]
    },
    {
        "title": "student",
        "company": "banking un